# Hubble expansion

*astro · beginner* · [open in Tensorizer](https://jackyko1991.github.io/tensorizer/#/astro/hubble_expansion)

Galaxies sit on a fixed comoving grid that is stretched by the scale factor $a(t)$. One outer product $x_{gi} = a\,q_{gi}$ moves every galaxy, and from any galaxy you choose the others recede with $v = H d$. A photon crossing the grid is stretched by the same factor: $1 + z = a_\mathrm{obs}/a_\mathrm{emit}$.

**Vector form**

$$\mathbf x_g = a(t)\,\mathbf q_g,\qquad \mathbf v_g - \mathbf v_o = H\,(\mathbf x_g - \mathbf x_o),\qquad H = \frac{\dot a}{a},\qquad 1 + z = \frac{a_\mathrm{obs}}{a_\mathrm{emit}}$$

**Index (tensor) form**

$$x_{tgi} = a_t\,q_{gi},\qquad v_{tgi} = H_t\,x_{tgi},\qquad H = \frac{v_{tgi}\,x_{tgi}}{x_{tgj}\,x_{tgj}}\Big|_{\text{rel. to } o}$$

| Symbol | Code | Shape | Meaning |
|---|---|---|---|
| $q_{gi}$ | `galaxy_grid(n, spacing)` | (G, 2) | comoving coordinates of galaxy $g$ (never change) |
| $a_t$ | `scale_factor(t, model, H0)` | (T,) | scale factor at every time, $a = 1$ today |
| $x_{tgi}$ | `einsum('t,gi->tgi', a, q)` | (T, G, 2) | physical positions: an outer product, no index summed |
| $H_t$ | `hubble_rate(t, model, H0)` | (T,) | $\dot a/a$: $2/(3t)$ for matter, constant for de Sitter |
| $v_{tgi}$ | `einsum('t,tgi->tgi', H, x)` | (T, G, 2) | Hubble-flow velocity of every galaxy |
| $x_{tgi} - x_{toi}$ | `x - x[:, o:o+1]` | (T, G, 2) | positions seen from observer $o$ (broadcast subtraction) |
| $v_{tgi}\,x_{tgi}$ | `einsum('tgi,tgi->t', v_rel, x_rel)` | (T,) | repeated $g, i$ ⇒ summed: least-squares slope of $v$ vs $d$ |
| $1 + z$ | `a / a[0]` | (T,) | stretch of the photon's wavelength since emission |

Units: Mpc, Gyr and km/s/Mpc. In the matter-dominated (Einstein–de Sitter) model $a = (t/t_0)^{2/3}$, so $H = 2/(3t)$ falls with time; in de Sitter $a = e^{H_0 (t - t_0)}$ and $H$ stays constant. Either way $v = Hd$ holds exactly for every observer: move the observer and the blue dots still fall on the same line as the open circles seen from the centre. The galaxies themselves do not move through space (no peculiar velocities): space between them grows. The photon covers comoving distance $c\,dt/a$ per step, and speeds above $c$ for distant galaxies are allowed, because nothing overtakes light locally.


## Setup

On Colab, choose **Runtime → Change runtime type → GPU** to run on a GPU — the code uses it automatically.

In [ ]:
import os
import subprocess
import sys

# Shared helpers (problems/_common) come from the repository; skip the clone if we are already inside it.
if not os.path.isdir("problems"):
    if not os.path.isdir("tensorizer"):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/jackyko1991/tensorizer.git"], check=True)
    os.chdir("tensorizer")
sys.path.insert(0, os.getcwd())
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyyaml", "tqdm"], check=True)

import torch

print("torch", torch.__version__, "· device:", "cuda" if torch.cuda.is_available() else "cpu")


## 1. `main.py` — main loop

In [ ]:
%%writefile problems/astro/hubble_expansion/main.py
"""Main loop: parameters → tensors → solve → plot."""

from pathlib import Path

from problems._common.runtime import cli, device

from . import equation, plot, solve, tensors


def run(params: dict, out_dir: Path) -> Path:
    dev = device()
    model = params["model"]
    H0 = tensors.hubble_constant(params["H0"])
    spacing = params["spacing"]
    q = tensors.galaxy_grid(params["n_grid"], spacing, dev)
    observer = tensors.nearest_galaxy(q, params["observer_x"] * spacing, params["observer_y"] * spacing)
    centre = tensors.nearest_galaxy(q, 0.0, 0.0)
    t_start = equation.time_of(params["a_start"], model, H0)
    t_end = equation.time_of(params["a_end"], model, H0)
    t = tensors.time_axis(t_start, t_end, 200, dev)

    sol = solve.solve(q, t, model, H0, observer)
    H_fit = solve.fitted_hubble_rate(sol["x_rel"], sol["v_rel"])

    print(f"{model}: today t0 = {equation.today(model, H0):.2f} Gyr, run from t = {t_start:.2f} to {t_end:.2f} Gyr")
    for k in (0, len(t) // 2, len(t) - 1):
        print(
            f"t = {t[k]:6.2f} Gyr  a = {sol['a'][k]:.3f}  H = ȧ/a = {sol['H'][k] * equation.KMS_PER_MPC_GYR:6.1f}"
            f"  fitted slope v/d from observer = {H_fit[k] * equation.KMS_PER_MPC_GYR:6.1f} km/s/Mpc"
        )
    print(f"photon emitted at a = {sol['a'][0]:.2f}, received at a = {sol['a'][-1]:.2f}: z = {sol['z'][-1]:.3f}")
    return plot.animate(sol, observer, centre, out_dir / "hubble_expansion.gif")


if __name__ == "__main__":
    cli(run, __file__)


## 2. `equation.py` — physics → tensor translation

In [ ]:
%%writefile problems/astro/hubble_expansion/equation.py
r"""Physics → tensor translation.

In an expanding universe galaxies keep fixed comoving coordinates q_{gi} (a grid that never changes) and
their physical positions grow with the scale factor a(t):

    x_{gi}(t) = a(t)\, q_{gi}

Differentiating in time gives every galaxy's velocity:

    v_{gi} = \dot a\, q_{gi} = H\, x_{gi},        H(t) = \dot a / a      (the Hubble rate)

Seen from any galaxy o (subtract its position and velocity), the same law holds:

    v_{gi} - v_{oi} = H\,(x_{gi} - x_{oi})

so every observer thinks they are at the centre of the expansion. Light also gets stretched with a(t):
a photon emitted at time t_e and received at t_o arrives redshifted by

    1 + z = \lambda_o / \lambda_e = a(t_o) / a(t_e).

On its way the photon covers comoving distance dχ = c\,dt / a(t).
"""

import math

import torch

C = 306.6  # speed of light in Mpc/Gyr
KMS_PER_MPC_GYR = 977.8  # 1 Mpc/Gyr = 977.8 km/s


def scale_factor(t: torch.Tensor, model: str, H0: float) -> torch.Tensor:
    r"""a(t), normalised to a = 1 today (t = t_0).   H0 in 1/Gyr.

    matter:     a = (t / t_0)^{2/3}   with t_0 = 2 / (3 H_0) the age of the universe
    de Sitter:  a = e^{H_0 (t - t_0)} with t_0 = 1 / H_0 (no Big Bang: just a label for "today")
    """
    if model == "matter":
        return (t / today(model, H0)) ** (2 / 3)
    return torch.exp(H0 * (t - today(model, H0)))


def time_of(a: float, model: str, H0: float) -> float:
    """Inverse of scale_factor: the time at which the universe had scale factor a."""
    t0 = today(model, H0)
    if model == "matter":
        return t0 * a**1.5
    return t0 + math.log(a) / H0


def today(model: str, H0: float) -> float:
    return 2 / (3 * H0) if model == "matter" else 1 / H0


def hubble_rate(t: torch.Tensor, model: str, H0: float) -> torch.Tensor:
    r"""H = \dot a / a: 2 / (3t) for matter, the constant H_0 for de Sitter."""
    if model == "matter":
        return 2 / (3 * t)
    return torch.full_like(t, H0)


def positions(q: torch.Tensor, a: torch.Tensor) -> torch.Tensor:
    r"""x_{tgi} = a_t q_{gi}: physical positions at every time.   q: (G, 2), a: (T,) → (T, G, 2)"""
    return torch.einsum("t,gi->tgi", a, q)


def velocities(x: torch.Tensor, H: torch.Tensor) -> torch.Tensor:
    r"""v_{tgi} = H_t x_{tgi} (Hubble flow).   x: (T, G, 2), H: (T,) → (T, G, 2)"""
    return torch.einsum("t,tgi->tgi", H, x)


def relative_to(field: torch.Tensor, o: int) -> torch.Tensor:
    """Positions or velocities as seen from galaxy o (subtract its own).   (T, G, 2) → (T, G, 2)"""
    return field - field[:, o : o + 1, :]


def photon_comoving_distance(t: torch.Tensor, a: torch.Tensor) -> torch.Tensor:
    r"""χ(t) = ∫ c\,dt' / a(t') from the first time step to t (trapezoid rule), (T,)."""
    integrand = C / a
    steps = 0.5 * (integrand[1:] + integrand[:-1]) * t.diff()
    return torch.cat([torch.zeros_like(t[:1]), steps.cumsum(0)])


def redshift(a_obs: torch.Tensor, a_emit: torch.Tensor) -> torch.Tensor:
    """z from 1 + z = a_obs / a_emit."""
    return a_obs / a_emit - 1


## 3. `tensors.py` — PyTorch tensors

In [ ]:
%%writefile problems/astro/hubble_expansion/tensors.py
"""Build the concrete tensors: the comoving galaxy grid and the time axis.

Units: Mpc for distances, Gyr for times, km/s/Mpc for H_0 in the parameters (converted to 1/Gyr here).
"""

import torch

from .equation import KMS_PER_MPC_GYR


def hubble_constant(H0_kms_mpc: float) -> float:
    """H_0 in 1/Gyr (70 km/s/Mpc ≈ 0.0716 /Gyr, so 1/H_0 ≈ 14 Gyr)."""
    return H0_kms_mpc / KMS_PER_MPC_GYR


def galaxy_grid(n: int, spacing: float, device: torch.device) -> torch.Tensor:
    """Comoving coordinates q_{gi} of an n × n grid centred on the origin: shape (n², 2), in Mpc."""
    k = (torch.arange(n, dtype=torch.float32) - (n - 1) / 2) * spacing  # (n,)
    qx, qy = torch.meshgrid(k, k, indexing="xy")
    return torch.stack([qx.flatten(), qy.flatten()], dim=-1).to(device)


def nearest_galaxy(q: torch.Tensor, x: float, y: float) -> int:
    """Index of the grid galaxy closest to the comoving point (x, y)."""
    target = torch.tensor([x, y], dtype=q.dtype, device=q.device)
    return int((q - target).norm(dim=-1).argmin())


def time_axis(t_start: float, t_end: float, n: int, device: torch.device) -> torch.Tensor:
    return torch.linspace(t_start, t_end, n, device=device)


## 4. `solve.py` — solver

In [ ]:
%%writefile problems/astro/hubble_expansion/solve.py
"""Evaluate the expansion on the whole time axis at once (no time stepping needed: a(t) is known)."""

import torch

from .equation import (
    hubble_rate,
    photon_comoving_distance,
    positions,
    redshift,
    relative_to,
    scale_factor,
    velocities,
)


def solve(q: torch.Tensor, t: torch.Tensor, model: str, H0: float, observer: int) -> dict:
    """Everything the plot needs, as tensors over time t (T,) and galaxies g (G,).

    A photon leaves an extra galaxy (the "emitter") at the first time step and reaches the observer
    exactly at the last one; the emitter's comoving distance is the photon's total comoving path.
    """
    a = scale_factor(t, model, H0)  # (T,)
    H = hubble_rate(t, model, H0)  # (T,)
    x = positions(q, a)  # (T, G, 2)
    v = velocities(x, H)  # (T, G, 2)

    chi = photon_comoving_distance(t, a)  # (T,) comoving distance covered by the photon
    q_obs = q[observer]
    q_emit = q_obs + torch.stack([chi[-1], torch.zeros_like(chi[-1])])  # emitter due "east" of the observer
    q_photon = q_emit - torch.einsum("t,i->ti", chi, torch.tensor([1.0, 0.0], dtype=q.dtype, device=q.device))
    return {
        "t": t,
        "a": a,
        "H": H,
        "x": x,
        "v": v,
        "x_rel": relative_to(x, observer),  # (T, G, 2) seen from the observer
        "v_rel": relative_to(v, observer),
        "emitter": torch.einsum("t,i->ti", a, q_emit),  # (T, 2)
        "photon": torch.einsum("t,ti->ti", a, q_photon),  # (T, 2)
        "z": redshift(a, a[0]),  # (T,) redshift the photon would show if it arrived at time t
    }


def fitted_hubble_rate(x_rel: torch.Tensor, v_rel: torch.Tensor) -> torch.Tensor:
    r"""Least-squares slope of v = H d at each time: H = Σ v_{gi} x_{gi} / Σ x_{gi} x_{gi}.   → (T,)"""
    return torch.einsum("tgi,tgi->t", v_rel, x_rel) / torch.einsum("tgi,tgi->t", x_rel, x_rel)


## 5. `plot.py` — 2D/3D plot → GIF

In [ ]:
%%writefile problems/astro/hubble_expansion/plot.py
"""Animate the expanding galaxy grid (seen from one observer) next to the velocity–distance plot."""

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from problems._common.plotting import ACCENT, frame_indices, save_gif

from .equation import KMS_PER_MPC_GYR


def animate(sol: dict, observer: int, centre: int, path: Path, arrow: float = 0.3) -> Path:
    """Velocity arrows are drawn as arrow · v / H, so their length is a fixed fraction of the distance to ★."""
    t, a, H = (sol[k].cpu().numpy() for k in ("t", "a", "H"))
    x = sol["x"].cpu().numpy()  # (T, G, 2)
    v_rel = sol["v_rel"].cpu().numpy()
    d_obs = np.linalg.norm(sol["x_rel"].cpu().numpy(), axis=-1)  # (T, G) distance from the observer
    s_obs = np.linalg.norm(v_rel, axis=-1) * KMS_PER_MPC_GYR / 1e3  # (T, G) speed in 1000 km/s
    x_c = x - x[:, centre : centre + 1]
    d_c = np.linalg.norm(x_c, axis=-1)
    s_c = np.linalg.norm(sol["v"].cpu().numpy() - sol["v"].cpu().numpy()[:, centre : centre + 1], axis=-1)
    s_c = s_c * KMS_PER_MPC_GYR / 1e3
    emitter, photon, z = sol["emitter"].cpu().numpy(), sol["photon"].cpu().numpy(), sol["z"].cpu().numpy()

    fig, (ax, ax_v) = plt.subplots(1, 2, figsize=(8, 4), gridspec_kw={"width_ratios": [1.1, 1]})
    reach = 1.05 * max(np.abs(x[-1]).max(), np.abs(emitter[-1]).max())
    ax.set_xlim(-reach, reach)
    ax.set_ylim(-reach, reach)
    ax.set_aspect("equal", adjustable="box")
    ax.set_xlabel("x (Mpc)")
    ax.set_ylabel("y (Mpc)")
    galaxies = ax.plot([], [], "o", ms=3.5, color="#6b7280")[0]
    arrows = ax.quiver(
        x[0, :, 0], x[0, :, 1], 0 * v_rel[0, :, 0], 0 * v_rel[0, :, 1],
        angles="xy", scale_units="xy", scale=1.0, width=0.004, color=ACCENT[1], alpha=0.7,
    )  # fmt: skip
    me = ax.plot([], [], "*", ms=13, color=ACCENT[0], zorder=5)[0]
    src = ax.plot([], [], "D", ms=6, color="#f59e0b", zorder=5)[0]
    trail = ax.plot([], [], "-", lw=1.5, color="#f59e0b")[0]
    light = ax.plot([], [], "o", ms=5, color="#b45309", zorder=6)[0]
    label = ax.text(0.03, 0.97, "", transform=ax.transAxes, va="top", fontsize=8)
    ax.set_title("★ observer   arrows ∝ velocity seen by ★   ◆ photon source", fontsize=8)

    d_max, s_max = 1.05 * d_obs.max(), 1.05 * max(s_obs.max(), s_c.max())
    ax_v.set_xlim(0, d_max)
    ax_v.set_ylim(0, s_max)
    ax_v.set_xlabel("distance d (Mpc)")
    ax_v.set_ylabel("recession speed v (1000 km/s)")
    pts_c = ax_v.plot([], [], "o", ms=6, mfc="none", color="#9ca3af", label="seen from the centre")[0]
    pts_o = ax_v.plot([], [], "o", ms=3, color=ACCENT[0], label="seen from ★")[0]
    law = ax_v.plot([], [], "--", color=ACCENT[1], lw=1.2, label="v = H d")[0]
    ax_v.legend(fontsize=8, loc="upper left")
    title = ax_v.set_title("", fontsize=9)

    def update(step: int):
        galaxies.set_data(x[step, :, 0], x[step, :, 1])
        arrows.set_offsets(x[step])
        arrows.set_UVC(arrow * v_rel[step, :, 0] / H[step], arrow * v_rel[step, :, 1] / H[step])
        me.set_data([x[step, observer, 0]], [x[step, observer, 1]])
        src.set_data([emitter[step, 0]], [emitter[step, 1]])
        path_now = a[step] / a[: step + 1, None] * photon[: step + 1]  # light's path so far, stretched to today
        trail.set_data(path_now[:, 0], path_now[:, 1])
        light.set_data([photon[step, 0]], [photon[step, 1]])
        label.set_text(f"t = {t[step]:.1f} Gyr   a = {a[step]:.2f}\nphoton stretched: 1 + z = {1 + z[step]:.2f}")
        pts_c.set_data(d_c[step], s_c[step])
        pts_o.set_data(d_obs[step], s_obs[step])
        H_kms = H[step] * KMS_PER_MPC_GYR
        law.set_data([0, d_max], [0, H_kms * d_max / 1e3])
        title.set_text(f"Hubble law: H = ȧ/a = {H_kms:.0f} km/s/Mpc")

    fig.tight_layout()
    return save_gif(fig, update, frame_indices(len(t), 90), path)


## Run

Change the parameters (or any file above) and run again.

In [ ]:
import importlib
from pathlib import Path

from IPython.display import Image, display

from problems._common.runtime import load_meta, resolve_params

params = {
    'model': 'matter',
    'H0': 70.0,
    'n_grid': 7,
    'spacing': 800.0,
    'observer_x': -2,
    'observer_y': -1,
    'a_start': 0.5,
    'a_end': 1.5
}

# Re-import so edits in the %%writefile cells above take effect.
for name in [m for m in sys.modules if m.startswith("problems.astro.hubble_expansion")]:
    del sys.modules[name]
main = importlib.import_module("problems.astro.hubble_expansion.main")
problem_dir = Path("problems/astro/hubble_expansion")
output = main.run(resolve_params(load_meta(problem_dir), params), Path("out"))
display(Image(filename=str(output)))
